In [114]:
import pandas as pd
import numpy as np

In [115]:
data = pd.read_csv('/home/hanya/breast-cancer.csv')

### Data prepocessing

In [116]:
data.dropna(axis=0)

,Class,Age,Meno-pause,Tumor-size,Inv-nodes,Node-caps,Deg-malig,Breast,Breast-quad,Irridiat
0,no-recurrence-events,30-39,premeno,30-34,0-2,no,3,left,left_low,no
1,no-recurrence-events,40-49,premeno,20-24,0-2,no,2,right,right_up,no
2,no-recurrence-events,40-49,premeno,20-24,0-2,no,2,left,left_low,no
3,no-recurrence-events,60-69,ge40,15-19,0-2,no,2,right,left_up,no
4,no-recurrence-events,40-49,premeno,0-4,0-2,no,2,right,right_low,no
...,...,...,...,...,...,...,...,...,...,...
272,recurrence-events,30-39,premeno,30-34,0-2,no,2,left,left_up,no
273,recurrence-events,30-39,premeno,20-24,0-2,no,3,left,left_up,yes
274,recurrence-events,60-69,ge40,20-24,0-2,no,1,right,left_up,no
275,recurrence-events,40-49,ge40,30-34,3-5,no,3,left,left_low,no


In [117]:
newdata = pd.get_dummies(data['Age'])
newdata = newdata.merge(pd.get_dummies(data['Meno-pause']),left_index=True,right_index=True)
newdata = newdata.merge(pd.get_dummies(data['Tumor-size']),left_index=True,right_index=True)
newdata = newdata.merge(pd.get_dummies(data['Inv-nodes']),left_index=True,right_index=True)
newdata = newdata.merge(pd.get_dummies(data['Node-caps']),left_index=True,right_index=True)
newdata = newdata.merge(pd.get_dummies(data['Deg-malig']),left_index=True,right_index=True)
newdata = newdata.merge(pd.get_dummies(data['Deg-malig']),left_index=True,right_index=True)
newdata = newdata.merge(pd.get_dummies(data['Breast']),left_index=True,right_index=True)
newdata = newdata.merge(pd.get_dummies(data['Breast-quad']),left_index=True,right_index=True)
newdata = newdata.merge(pd.get_dummies(data['Irridiat']),left_index=True,right_index=True)
newdata = newdata.merge(data['Class'],left_index=True,right_index=True)
newdata['Class'] = data['Class'].apply({'recurrence-events':1, 'no-recurrence-events':0}.get) 
newdata.to_csv('newdata.csv')

## Cross Validation

In [118]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=10, shuffle=True, random_state=None)

for k, (train, test) in enumerate(kf.split(data)):
    data.iloc[train].to_csv('train' + str(k+1) + '.csv', index=False, header=False)
    data.iloc[test].to_csv('test'+str(k+1) + '.csv', index=False, header=False)

#### Composing on Plus and Minus context

In [119]:
def load(i):
    train = pd.read_csv('/home/hanya/train' + str(i) + '.csv' , sep=',', header=None)
    plus = train[train[0]==1]
    minus = train[train[0]==0]
    unknown = pd.read_csv('/home/hanya/test' + str(i) + '.csv' , sep=',', header=None)
    
    return np.array(plus), np.array(minus), np.array(unknown)

In [120]:
def make_intent(example):
    return set([i + ':' + str(k) for i, k in zip(attrib_names, example) if k])

In [121]:
def classify(context_plus, context_minus, example):
    a = 0; b = 0
    eintent = make_intent(example)
    eintent.discard('Class:1')
    eintent.discard('Class:0')
    labels = {"positive": 0, "negative": 0}
    for e in context_plus:
        ei = make_intent(e)
        candidate_intent = ei & eintent
        a +=len(candidate_intent)
    for e in context_minus:
        ei = make_intent(e)
        candidate_intent = ei & eintent
        b +=len(candidate_intent)
    if a > b:
        if example[-1]:
            return "TP"
        return "FP"
    
    else:
        if example[-1]:
            return "FN"
        return "TN"

In [122]:
(iplus, iminus, iunknown) = load(6)
acc = 0
prec = 0
rec = 0
cv_res = {
        "TP": 0,
        "TN": 0,
        "FP": 0,
        "FN": 0,
        "contradictory": 0,
        }
for elem in iunknown:
    pin = classify(iplus, iminus, elem)
    
    cv_res[pin] += 1
res = results(cv_res)
acc += res['accuracy']
prec += res['precision']
rec += res['recall']
print(acc)


0.8214285714285714


In [123]:
def classify1(context_plus, context_minus, example):
    a = 0; b = 0
    eintent = make_intent(example)
    eintent.discard('Class:1')
    eintent.discard('Class:0')
    labels = {"positive": 0, "negative": 0}
    for e in context_plus:
        ei = make_intent(e)
        candidate_intent = ei & eintent
        if candidate_intent:
            a += 1
    for e in context_minus:
        ei = make_intent(e)
        candidate_intent = ei & eintent
        if candidate_intent:
            b += 1
    if a > b:
        if example[-1]:
            return "TP"
        return "FP"
    
    else:
        if example[-1]:
            return "FN"
        return "TN"

In [124]:
(iplus, iminus, iunknown) = load(4)
acc = 0
prec = 0
rec = 0
cv_res = {
        "TP": 0,
        "TN": 0,
        "FP": 0,
        "FN": 0,
        "contradictory": 0,
        }
for elem in iunknown:
    pin = classify1(iplus, iminus, elem)
    
    cv_res[pin] += 1
res = results(cv_res)
acc += res['accuracy']
prec += res['precision']
rec += res['recall']
print(acc)

0.7857142857142857
